# 02 - Cobertura de Transporte por Bairro

Cruza as paradas de ônibus com os bairros de Recife usando GeoPandas.
Calcula a **densidade de paradas por km²** e gera uma **nota de mobilidade (0 a 10)** para cada bairro.

**Fluxo:**
1. Carrega o shapefile dos bairros (do Rodrigo)
2. Carrega as paradas (do notebook anterior)
3. Faz o cruzamento espacial: conta paradas dentro de cada bairro
4. Calcula densidade e normaliza para nota 0-10
5. Salva a tabela final

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt

In [ ]:
# Localiza a raiz do projeto
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()

ROOT = find_root()

# Caminhos dos arquivos de entrada
BAIRROS_PATH = ROOT / 'data' / 'processed' / 'recife_renda.geojson'   # do Rodrigo
PARADAS_PATH = ROOT / 'data' / 'processed' / 'recife_paradas.geojson' # do notebook 01

print(f'ROOT: {ROOT}')
print(f'Bairros: {BAIRROS_PATH} | existe: {BAIRROS_PATH.exists()}')
print(f'Paradas: {PARADAS_PATH} | existe: {PARADAS_PATH.exists()}')

In [ ]:
# Carrega os bairros de Recife (gerado pelo Rodrigo — dados por setor censitário)
bairros = gpd.read_file(BAIRROS_PATH)
print(f'Setores carregados: {len(bairros)}')
print('CRS:', bairros.crs)

In [ ]:
# Identifica a coluna com o NOME do bairro (prefere NM_ sobre CD_)
col_bairro = None
for col in bairros.columns:
    if col.upper().startswith('NM_') and 'BAIRRO' in col.upper():
        col_bairro = col
        break
if col_bairro is None:
    for col in bairros.columns:
        if 'bairro' in col.lower():
            col_bairro = col
            break

print(f'Coluna do nome do bairro: "{col_bairro}"')
print('Exemplos:', bairros[col_bairro].dropna().unique()[:10])

# O arquivo do Rodrigo tem setores censitários (vários por bairro)
# Precisamos dissolver em um único polígono por bairro
print(f'\nLinhas antes do dissolve (setores): {len(bairros)}')
bairros = bairros[[col_bairro, 'geometry']].dropna(subset=[col_bairro])
bairros = bairros.dissolve(by=col_bairro).reset_index()
print(f'Linhas depois do dissolve (bairros): {len(bairros)}')

In [ ]:
# Garante que ambos os GeoDataFrames usam o mesmo sistema de coordenadas
paradas = gpd.read_file(PARADAS_PATH)
print(f'Paradas carregadas: {len(paradas)}')

# Reprojecta para SIRGAS 2000 (EPSG:4674) se necessário
if bairros.crs != paradas.crs:
    paradas = paradas.to_crs(bairros.crs)
    print(f'Paradas reprojetadas para: {bairros.crs}')
else:
    print('CRS já compatível:', bairros.crs)

In [ ]:
# CRUZAMENTO ESPACIAL: descobre em qual bairro está cada parada
# sjoin = spatial join — como um JOIN de banco de dados, mas usando geometria
print('Fazendo cruzamento espacial (spatial join)...')
paradas_no_bairro = gpd.sjoin(
    paradas,        # pontos (paradas)
    bairros[[col_bairro, 'geometry']],  # polígonos (bairros)
    how='inner',    # mantém só paradas que caem dentro de algum bairro
    predicate='within'
)

print(f'Paradas dentro dos bairros: {len(paradas_no_bairro)}')
display(paradas_no_bairro[[col_bairro, 'stop_name', 'geometry']].head(5))

In [ ]:
# Conta quantas paradas há em cada bairro
contagem = (
    paradas_no_bairro
    .groupby(col_bairro)
    .size()
    .reset_index(name='num_paradas')
)

print(f'Bairros com pelo menos uma parada: {len(contagem)}')
print('\nTop 10 bairros com mais paradas:')
display(contagem.sort_values('num_paradas', ascending=False).head(10))

In [ ]:
# Junta a contagem de volta ao GeoDataFrame dos bairros
bairros_mob = bairros.merge(contagem, on=col_bairro, how='left')

# Bairros sem nenhuma parada recebem 0
bairros_mob['num_paradas'] = bairros_mob['num_paradas'].fillna(0).astype(int)

print(f'Total de bairros: {len(bairros_mob)}')
print(f'Bairros sem paradas: {(bairros_mob["num_paradas"] == 0).sum()}')
display(bairros_mob[[col_bairro, 'num_paradas']].head(10))

In [ ]:
# Calcula a ÁREA de cada bairro em km²
# Reprojecta para sistema métrico (SIRGAS / UTM zona 25S) para calcular área corretamente
bairros_utm = bairros_mob.to_crs('EPSG:31985')
bairros_mob['area_km2'] = bairros_utm.geometry.area / 1_000_000  # m² → km²

# Calcula densidade: paradas por km²
bairros_mob['densidade_paradas'] = bairros_mob['num_paradas'] / bairros_mob['area_km2'].replace(0, np.nan)
bairros_mob['densidade_paradas'] = bairros_mob['densidade_paradas'].fillna(0)

print('Estatísticas de densidade (paradas/km²):')
display(bairros_mob['densidade_paradas'].describe().round(2))

In [ ]:
# Normaliza para nota de 0 a 10 (min-max scaling)
# Bairro com maior densidade → nota 10 | sem paradas → nota 0
d_min = bairros_mob['densidade_paradas'].min()
d_max = bairros_mob['densidade_paradas'].max()

bairros_mob['nota_mobilidade'] = (
    (bairros_mob['densidade_paradas'] - d_min) / (d_max - d_min) * 10
).round(2)

print('Distribuição das notas de mobilidade:')
display(bairros_mob['nota_mobilidade'].describe().round(2))

print('\nTop 10 bairros com melhor mobilidade:')
display(
    bairros_mob[[col_bairro, 'num_paradas', 'area_km2', 'densidade_paradas', 'nota_mobilidade']]
    .sort_values('nota_mobilidade', ascending=False)
    .head(10)
    .round(3)
)

In [ ]:
# Gráfico de barras dos 15 piores e 15 melhores bairros
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bairros_validos = bairros_mob.dropna(subset=[col_bairro]).copy()
bairros_validos[col_bairro] = bairros_validos[col_bairro].astype(str)

top15 = bairros_validos.nlargest(15, 'nota_mobilidade')
bot15 = bairros_validos.nsmallest(15, 'nota_mobilidade')

axes[0].barh(top15[col_bairro], top15['nota_mobilidade'], color='steelblue')
axes[0].set_title('15 Bairros com MELHOR Mobilidade', fontsize=12)
axes[0].set_xlabel('Nota (0-10)')
axes[0].invert_yaxis()

axes[1].barh(bot15[col_bairro], bot15['nota_mobilidade'], color='salmon')
axes[1].set_title('15 Bairros com PIOR Mobilidade', fontsize=12)
axes[1].set_xlabel('Nota (0-10)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Monta a tabela final no formato pedido pelo Rodrigo para integração no IVU
tabela_final = bairros_mob[[col_bairro, 'nota_mobilidade', 'num_paradas']].copy()
tabela_final.columns = ['bairro', 'nota_dimensao', 'dado_principal']
tabela_final['dado_principal'] = tabela_final['dado_principal'].astype(str) + ' paradas'

display(tabela_final.head(10))

# Salva a tabela CSV para o Rodrigo
OUT_CSV = ROOT / 'data' / 'processed' / 'mobilidade_por_bairro.csv'
tabela_final.to_csv(OUT_CSV, index=False)
print(f'Tabela salva em: {OUT_CSV}')

# Salva o GeoDataFrame completo para o mapa
OUT_GEO = ROOT / 'data' / 'processed' / 'recife_mobilidade.geojson'
bairros_mob.to_file(OUT_GEO, driver='GeoJSON')
print(f'GeoJSON salvo em: {OUT_GEO}')